In [383]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd

from torch import Tensor
from typing import *

from torch_scatter import scatter
from torch_geometric.utils import softmax

import triton
import triton.language as tl

from copy import deepcopy

In [319]:
# class SoftmaxSumFunction(autograd.Function):
#     @staticmethod
#     def forward(
#         ctx: autograd.function.FunctionCtx,
#         x: Tensor,
#         a: Tensor,
#         _x: Tensor,
#         index: Tensor,
#         mask: Tensor,
#         exp_a: Tensor,
#         sum_exp_a: Tensor,
#         sum_exp_ah: Tensor,
#         training: bool,
#     ):
#         dlt_exp_ah = exp_a[mask, None] * (x - _x)
#         sum_exp_ah = scatter(dlt_exp_ah, index, dim=0, out=sum_exp_ah.clone(), reduce="sum")
#         out = sum_exp_ah / sum_exp_a[:, None]
#         if training:
#             w = exp_a[mask] / sum_exp_a.index_select(0, index)
#             ctx.save_for_backward(out, x, w, index)
#         return out
    
#     @staticmethod
#     def backward(
#         ctx: autograd.function.FunctionCtx,
#         grad: Tensor,
#     ):
#         out, x, w, index = ctx.saved_tensors
#         g = grad.index_select(0, index)
#         o =  out.index_select(0, index)

#         grad_x = g * w[:, None]
#         grad_a = (g * (x - o)).sum(dim=-1) * w
#         return grad_x, grad_a, None, None, None, None, None, None, None

In [320]:
# class SoftmaxSum(nn.Module):
#     def __init__(self,
#         num_nodes: int,
#         num_edges: int,
#         num_features: int,
#         dtype: Optional[torch.dtype] = None,
#     ) -> None:
#         super().__init__()
#         self.num_nodes = num_nodes
#         self.num_edges = num_edges

#         self.register_buffer("max_hist_a", torch.zeros(num_nodes, dtype=dtype))
#         self.register_buffer("exp_hist_a", torch.zeros(num_edges, dtype=dtype))
#         self.register_buffer("sum_exp_a", torch.zeros(num_nodes, dtype=dtype))
#         self.register_buffer("sum_exp_ah", torch.zeros(num_nodes, num_features, dtype=dtype))
    
#     @torch.no_grad()
#     def update_snapshot(self, x: Tensor, a: Tensor, index: Tensor):
#         max_a = scatter(a, index, dim=0, dim_size=self.num_nodes, reduce="max")
#         exp_a = (a - max_a.index_select(0, index)).exp_()
#         self.get_buffer("max_hist_a")[:] = max_a
#         self.get_buffer("exp_hist_a")[:] = exp_a

#         sum_exp_a = scatter(exp_a, index, dim=0, dim_size=self.num_nodes, reduce="sum") + 1e-10
#         sum_exp_ah = scatter(exp_a[:,None] * x, index, dim=0, dim_size=self.num_nodes, reduce="sum")
#         self.get_buffer("sum_exp_a")[:] = sum_exp_a
#         self.get_buffer("sum_exp_ah")[:] = sum_exp_ah
#         return self
    
#     @torch.no_grad()
#     def update_attention(self, x: Tensor, a: Tensor, _x: Tensor, index: Tensor, mask: Tensor):
#         exp_a = (a - self.get_buffer("max_hist_a").index_select(0, index)).exp_()
#         exp_ah = exp_a[:, None] * x

#         exp_hist_a = self.get_buffer("exp_hist_a")[mask]
#         exp_hist_ah = exp_hist_a[:, None] * _x

#         self.get_buffer("exp_hist_a")[mask] = exp_a
#         scatter(exp_a - exp_hist_a, index, dim=0, out=self.get_buffer("sum_exp_a"), reduce="sum")
#         scatter(exp_ah - exp_hist_ah, index, dim=0, out=self.get_buffer("sum_exp_ah"), reduce="sum")
#         return self
    
#     def forward(self, x: Tensor, a: Tensor, _x: Tensor, index: Tensor, mask: Tensor):
#         return SoftmaxSumFunction.apply(x, a, _x, index, mask,
#                                  self.get_buffer("exp_hist_a"),
#                                  self.get_buffer("sum_exp_a"),
#                                  self.get_buffer("sum_exp_ah"),
#                                  self.training)

In [321]:
# num_nodes = 200
# num_edges = 100
# num_features = 8
# for i in range(10):
#     x = torch.randn(num_edges, num_features)
#     a = torch.randn(num_edges)
#     index = torch.randint(num_nodes, size=(num_edges,), dtype=torch.long)
#     mask = torch.ones(num_edges, dtype=torch.bool)

#     ss = SoftmaxSum(num_nodes, num_edges, num_features)
#     ss.update_snapshot(x, a, index)
#     ss.update_attention(x, a, x, index, mask)

#     x.requires_grad_()
#     a.requires_grad_()
#     ss(x, a, x, index, mask).sum().backward()
#     grad_x0 = x.grad.clone()
#     grad_a0 = a.grad.clone()

#     x.grad = None
#     a.grad = None
#     t = softmax(a, index, num_nodes=num_nodes)[:,None] * x
#     scatter(t, index, dim=0, dim_size=num_nodes, reduce="sum").sum().backward()

#     # print(torch.norm(grad_x0 - x.grad), torch.norm(grad_a0 - a.grad))
#     # print(torch.abs(grad_x0 - x.grad).max(), torch.abs(grad_a0 - a.grad).max())
#     # print(grad_x0)
#     # print(grad_a0)
#     print(grad_a0 - a.grad)

In [355]:
class SoftmaxSumFunction(autograd.Function):
    @staticmethod
    def forward(
        ctx: autograd.function.FunctionCtx,
        x1: Tensor,
        a1: Tensor,
        x0: Tensor,
        a0: Tensor,
        index: Tensor,
        max_a: Tensor,
        sum_exp_a: Tensor,
        sum_exp_ah: Tensor,
        training: bool,
    ):
        exp_a = (a0 - max_a.index_select(0, index)).exp_()
        dlt_exp_ah = exp_a[:, :, None] * (x1 - x0)
        sum_exp_ah = scatter(dlt_exp_ah, index, dim=0, out=sum_exp_ah.clone(), reduce="sum")
        out = sum_exp_ah / sum_exp_a[:, :, None]
        if training:
            w = exp_a / sum_exp_a.index_select(0, index)
            ctx.save_for_backward(out, x1, w, index)
        return out
    
    @staticmethod
    def backward(
        ctx: autograd.function.FunctionCtx,
        grad: Tensor,
    ):
        out, x, w, index = ctx.saved_tensors
        g = grad.index_select(0, index)
        o =  out.index_select(0, index)

        grad_x = g * w[:, :, None]
        grad_a = (g * (x - o)).sum(dim=-1) * w
        return grad_x, grad_a, None, None, None, None, None, None, None

class SoftmaxSum(nn.Module):
    def __init__(self,
        num_nodes: int,
        num_edges: int,
        num_heads: int,
        num_features: int,
        dtype: Optional[torch.dtype] = None,
    ) -> None:
        super().__init__()
        self.num_nodes = num_nodes
        self.num_edges = num_edges
        self.num_heads = num_heads

        self.register_buffer("max_a", torch.zeros(num_nodes, num_heads, dtype=dtype))
        self.register_buffer("sum_exp_a", torch.zeros(num_nodes, num_heads, dtype=dtype))
        self.register_buffer("sum_exp_ah", torch.zeros(num_nodes, num_heads, num_features, dtype=dtype))
    
    @torch.no_grad()
    def update_snapshot(self, x: Tensor, a: Tensor, index: Tensor):
        max_a = scatter(a, index, dim=0, dim_size=self.num_nodes, reduce="max")
        exp_a = (a - max_a.index_select(0, index)).exp_()
        self.get_buffer("max_a")[:] = max_a

        sum_exp_a = scatter(exp_a, index, dim=0, dim_size=self.num_nodes, reduce="sum") + 1e-10
        sum_exp_ah = scatter(exp_a[:, :, None] * x, index, dim=0, dim_size=self.num_nodes, reduce="sum")
        self.get_buffer("sum_exp_a")[:] = sum_exp_a
        self.get_buffer("sum_exp_ah")[:] = sum_exp_ah
        return self
    
    @torch.no_grad()
    def update_attention(self, x1: Tensor, a1: Tensor, x0: Tensor, a0: Tensor, index: Tensor):
        max_hist_a = self.get_buffer("max_a").index_select(0, index)
        exp_a0 = (a0 - max_hist_a).exp_()
        exp_a1 = (a1 - max_hist_a).exp_()
        scatter(exp_a1 - exp_a0, index, dim=0, out=self.get_buffer("sum_exp_a"), reduce="sum")

        exp_ah0 = exp_a0[:, :, None] * x0
        exp_ah1 = exp_a1[:, :, None] * x1
        scatter(exp_ah1 - exp_ah0, index, dim=0, out=self.get_buffer("sum_exp_ah"), reduce="sum")
        return self
    
    def forward(self, x1: Tensor, a1: Tensor, x0: Tensor, a0: Tensor, index: Tensor) -> Tensor:
        return SoftmaxSumFunction.apply(x1, a1, x0, a0, index,
                                 self.get_buffer("max_a"),
                                 self.get_buffer("sum_exp_a"),
                                 self.get_buffer("sum_exp_ah"),
                                 self.training)

In [359]:
num_nodes = 200
num_edges = 100
num_heads = 2
num_features = 256
for i in range(10):
    x = torch.randn(num_edges, num_heads, num_features)
    a = torch.randn(num_edges, num_heads)
    index = torch.randint(num_nodes, size=(num_edges,), dtype=torch.long)

    ss = SoftmaxSum(num_nodes, num_edges, num_heads, num_features)
    ss.update_snapshot(x, a, index)
    ss.update_attention(x, a, x, a, index)

    x.requires_grad_()
    a.requires_grad_()
    ss(x, a, x, a, index).sum().backward()
    grad_x0 = x.grad.clone()
    grad_a0 = a.grad.clone()

    x.grad = None
    a.grad = None
    t = softmax(a, index, num_nodes=num_nodes)[:, :, None] * x
    scatter(t, index, dim=0, dim_size=num_nodes, reduce="sum").sum().backward()

    print(torch.norm(grad_x0 - x.grad), torch.norm(grad_a0 - a.grad))
    # print(torch.abs(grad_x0 - x.grad).max(), torch.abs(grad_a0 - a.grad).max())
    # print(grad_x0)
    print(grad_a0[:8, :])
    print(a.grad[:8, :])
    # print(grad_a0 - a.grad)

tensor(0.) tensor(6.2413e-06)
tensor([[-2.1367, -3.5242],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.8726, -6.0982]])
tensor([[-2.1367, -3.5242],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.8726, -6.0982]])
tensor(0.) tensor(7.5973e-06)
tensor([[ 4.1189, -1.3866],
        [ 6.2586, -4.8426],
        [ 0.0000,  0.0000],
        [ 2.7955,  8.4197],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000]])
tensor([[ 4.1189, -1.3866],
        [ 6.2586, -4.8426],
        [ 0.0000,  0.0000],
        [ 2.7955,  8.4197],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000],
        [ 0.0000,  0.0000]])
tensor(0.) tensor(6.0887e-06)
tensor([[ 

In [399]:
class GATConv(nn.Module):
    def __init__(self,
        in_channels: int,
        out_channels: int,
        heads: int = 1,
        concat: bool = False,
        negative_slope: float = 0.2,
        bias: bool = True,
        **kwargs
    ) -> None:
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.negative_slope = negative_slope
        self._softmax_sum_list = [None]
        
        self.weight = nn.Parameter(torch.Tensor(in_channels, heads * out_channels))
        self.att_src = nn.Parameter(torch.Tensor(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.Tensor(1, heads, out_channels))
        
        if bias and concat:
            self.bias = nn.Parameter(torch.Tensor(heads * out_channels))
        elif bias and not concat:
            self.bias = nn.Parameter(torch.Tensor(out_channels))
        else:
            self.bias = None

        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_normal_(self.weight)
        nn.init.xavier_normal_(self.att_src)
        nn.init.xavier_normal_(self.att_dst)
        if self.bias is not None:
            nn.init.zeros_(self.bias)
    
    @property
    def softmax_sum(self) -> Optional[SoftmaxSum]:
        return self._softmax_sum_list[0]
    
    def set_softmax_sum(self, m: Optional[SoftmaxSum]):
        self._softmax_sum_list = [m]
        return self
    
    def train(self, mode: bool = True):
        if self.softmax_sum is not None:
            self.softmax_sum.train(mode)
        return super().train(mode)
    
    def update_softmax_snapshot(self,
        x: Tensor,
        edge_index: Tensor,
        snap_net: nn.Module,
    ):
        snap_net: GATConv = snap_net.eval()
        with torch.no_grad():
            x, a = snap_net._att_x(x, edge_index)
        self.softmax_sum.update_snapshot(x, a, edge_index[1])

    def _att_x(self, x: Tensor, edge_index: Tensor) -> Tuple[Tensor, Tensor]:
        H, C = self.heads, self.out_channels
        x = (x @ self.weight).view(-1, H, C)

        src_a = (x * self.att_src).sum(dim=-1)
        src_a = src_a[edge_index[0]]
        dst_a = (x * self.att_dst).sum(dim=-1)
        dst_a = dst_a[edge_index[1]]

        x = x[edge_index[0]]
        a = F.leaky_relu(src_a + dst_a, self.negative_slope)
        return x, a
    
    def forward(self,
        x1: Tensor,
        x0: Tensor,
        edge_index: Tensor,
        snap_net: nn.Module,
        update_att: bool,
    ) -> Tensor:
        H, C = self.heads, self.out_channels
        snap_net: GATConv = snap_net.eval()

        index = edge_index[1]
        with torch.no_grad():
            x0, a0 = snap_net._att_x(x0, edge_index)
            if update_att:
                x, a = snap_net._att_x(x1, edge_index)
                self.softmax_sum.update_attention(x, a, x0, a0, index)
                x0, a0 = x, a
        
        x1, a1 = self._att_x(x1, edge_index)
        out = self.softmax_sum.forward(x1, a1, x0, a0, index)
        
        if self.concat:
            out = out.view(-1, H * C)
        else:
            out = out.mean(dim=1)

        if self.bias is not None:
            out += self.bias
        return out
        

In [400]:
@torch.no_grad()
def slim_local_graph(src_mask: Tensor, edge_index: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
    edge_mask = src_mask[edge_index[0]]
    slim_edge_index = edge_index[:, edge_mask]
    
    node_mask = src_mask.clone().index_fill_(0, slim_edge_index[1], 1)

    idx = torch.where(node_mask)[0]
    imp = torch.empty_like(src_mask, dtype=torch.long).fill_((2**62-1)*2+1)
    imp[idx] = torch.arange(idx.size(0))
    
    slim_edge_index = imp[slim_edge_index.flatten()].view_as(slim_edge_index)
    return node_mask, edge_mask, slim_edge_index

In [401]:
src_mask = torch.rand(num_nodes) < 0.3
edge_index = torch.randint(num_nodes, size=(2, num_edges), dtype=torch.long)
node_mask, edge_mask, slim_edge_index = slim_local_graph(src_mask, edge_index)
print(node_mask.size())
print(edge_mask.size())
print(slim_edge_index.size())

torch.Size([200])
torch.Size([100])
torch.Size([2, 35])


In [484]:
net = GATConv(3, num_features, num_heads)
att = SoftmaxSum(num_nodes, num_edges, num_heads, num_features)
net.set_softmax_sum(att)

snap_net = deepcopy(net)

In [485]:
x1 = torch.randn(num_nodes, 3)
x0 = torch.randn(num_nodes, 3)
edge_index = torch.randint(num_nodes, size=(2, num_edges), dtype=torch.long)

net.update_softmax_snapshot(x0, edge_index, snap_net)

In [511]:
x = net.forward(x1, x0, edge_index, snap_net, update_att=True)
x0 = x1
x

tensor([[ 0.1482, -0.0744, -0.0213,  ..., -0.0421, -0.1849, -0.1039],
        [-0.0075,  0.0321, -0.0046,  ...,  0.0154,  0.0506, -0.0110],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0878, -0.0887, -0.0550,  ..., -0.0982, -0.1763, -0.0175],
        [ 0.1193,  0.0695, -0.0637,  ...,  0.0060,  0.0391, -0.1508],
        [ 0.0294,  0.0186,  0.0140,  ...,  0.0327,  0.0128, -0.0487]],
       grad_fn=<AddBackward0>)